<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRYOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.1/949.1 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
import os
import numpy as np
import cv2
from ultralytics import YOLO
import random
import torch.nn.functional as F

In [4]:
# Süper Çözünürlük Modülü
class SuperResolution(nn.Module):
    def __init__(self):
        super(SuperResolution, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

In [40]:
# Başlangıçta import edilmeli
from ultralytics import YOLO

# SRYOLO Modeli
class SRYOLO(nn.Module):
    def __init__(self, yolo_weights):
        super(SRYOLO, self).__init__()
        self.sr = SuperResolution()
        self.yolo = YOLO(yolo_weights)
        self.sr_loss_fn = nn.MSELoss()  # SR eğitimi için kayıp fonksiyonu

    def forward(self, x):
        # 1. Süper çözünürlük aşaması
        sr_output = self.sr(x)

        # 2. YOLOv8 ile nesne tespiti - YOLO farklı çağrılır
        with torch.no_grad():  # Eğitim sırasında YOLO'yu dondur
            results = self.yolo.predict(sr_output)

        return sr_output, results

    def compute_sr_loss(self, sr_output, high_res_target):
        return self.sr_loss_fn(sr_output, high_res_target)

    def train_sr(self, mode=True):
        self.sr.train(mode)  # Sadece SR modülünü eğit
        return self  # Metot zincirlemesi için self döndür

In [51]:
# Görüntü İşleme Fonksiyonları
def apply_dark_channel_prior(img):
    """Dark Channel Prior algoritması uygulama"""
    if isinstance(img, torch.Tensor):
        # Tensörü numpy dizisine dönüştür
        np_img = img.permute(1, 2, 0).cpu().numpy()
    else:
        np_img = np.array(img) / 255.0

    # RGB formatından BGR formatına dönüştür (OpenCV için)
    if np_img.shape[2] == 3:
        np_img = np_img[:, :, ::-1]

    # Görüntüyü normalize et
    np_img = np.clip(np_img, 0.0, 1.0)

    # Görüntünün her pikseli için minimum değeri bul
    min_channel = np.min(np_img, axis=2)

    # Minimum değerler üzerinde filtre uygula
    kernel_size = 15
    dark_channel = cv2.erode(min_channel, np.ones((kernel_size, kernel_size)))

    # Atmospherik ışık tahmini
    num_pixels = dark_channel.size
    num_brightest = int(0.001 * num_pixels)
    flat_dark = dark_channel.flatten()
    flat_img = np_img.reshape(-1, 3)
    indices = np.argsort(flat_dark)[-num_brightest:]
    atmospheric = np.mean(flat_img[indices], axis=0)

    # İletim haritası tahmini
    omega = 0.95
    transmission = 1 - omega * dark_channel / np.max(atmospheric)

    # İletim haritasını düzelt
    transmission = cv2.GaussianBlur(transmission, (kernel_size, kernel_size), 0)
    transmission = np.clip(transmission, 0.1, 1.0)

    # Bulanıklığı gider
    result = np.zeros_like(np_img)
    for i in range(3):
        result[:, :, i] = (np_img[:, :, i] - atmospheric[i]) / transmission + atmospheric[i]

    # Sonucu 0-1 aralığında sınırla
    result = np.clip(result, 0.0, 1.0)

    # BGR'dan RGB'ye dönüştür
    if result.shape[2] == 3:
        result = result[:, :, ::-1]

    # Gerekirse tensöre dönüştür
    if isinstance(img, torch.Tensor):
        result = result.copy()
        result = torch.from_numpy(result).permute(2, 0, 1).float()

    return (result * 255).cpu().numpy().astype(np.uint8)

In [7]:
def apply_clahe(img):
    """CLAHE (Contrast Limited Adaptive Histogram Equalization) uygulama"""
    if isinstance(img, torch.Tensor):
        # Tensörü numpy dizisine dönüştür
        np_img = img.permute(1, 2, 0).cpu().numpy()
        np_img = (np_img * 255).astype(np.uint8)
    else:
        np_img = np.array(img)

    # BGR'a dönüştür (OpenCV için)
    if np_img.shape[2] == 3:
        np_img = cv2.cvtColor(np_img, cv2.COLOR_RGB2BGR)

    # LAB renk uzayına dönüştür
    lab = cv2.cvtColor(np_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # CLAHE oluştur ve uygula
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    # Kanalları birleştir
    enhanced_lab = cv2.merge((cl, a, b))

    # BGR'a geri dönüştür
    enhanced_bgr = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

    # RGB'ye dönüştür
    enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)

    # Gerekirse tensöre dönüştür
    if isinstance(img, torch.Tensor):
        enhanced_rgb = torch.from_numpy(enhanced_rgb).permute(2, 0, 1).float() / 255.0
        return enhanced_rgb

    return enhanced_rgb

In [8]:
def downsample_image(img, scale_factor=0.5):
    """Görüntüyü düşük çözünürlüğe dönüştür"""
    if isinstance(img, torch.Tensor):
        # PyTorch tensörleri için yeniden boyutlandırma
        h, w = img.shape[-2], img.shape[-1]
        new_h, new_w = int(h * scale_factor), int(w * scale_factor)
        down_img = F.interpolate(img.unsqueeze(0), size=(new_h, new_w), mode='bicubic', align_corners=False).squeeze(0)
        # Orijinal boyutuna geri getir
        up_img = F.interpolate(down_img.unsqueeze(0), size=(h, w), mode='bicubic', align_corners=False).squeeze(0)
        return up_img
    else:
        # PIL görüntüleri için yeniden boyutlandırma
        w, h = img.size
        new_w, new_h = int(w * scale_factor), int(h * scale_factor)
        down_img = img.resize((new_w, new_h), Image.BICUBIC)
        # Orijinal boyutuna geri getir
        up_img = down_img.resize((w, h), Image.BICUBIC)
        return up_img

In [54]:
# Özel veri seti sınıfı
class SRYOLODataset(Dataset):
    def __init__(self, image_dir, labels_dir, transform=None, augment=True, scale_factor=0.5):
        self.image_dir = image_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.augment = augment
        self.scale_factor = scale_factor
        # Görüntü dosyalarını listele
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
      img_name = self.image_files[idx]
      img_id = os.path.splitext(img_name)[0]

      # Orijinal görüntüyü yükle
      img_path = os.path.join(self.image_dir, img_name)
      original_img = Image.open(img_path).convert('RGB')

      # Etiket dosyasını yükle (YOLOv8 formatında)
      label_path = os.path.join(self.labels_dir, f"{img_id}.txt")

      # Etiketleri doğrudan oku ve bir tensör veya liste olarak döndür
      if os.path.exists(label_path):
          with open(label_path, 'r') as f:
              labels_raw = f.readlines()

          # YOLO formatındaki etiketleri (class, x, y, width, height) işle
          labels = []
          for label in labels_raw:
              # Her bir satırı boşluklara göre böl ve float'a çevir
              label_parts = [float(x) for x in label.strip().split()]
              labels.append(label_parts)

          # Etiketleri tensör olarak dönüştür
          labels = torch.tensor(labels) if labels else torch.zeros((0, 5))
      else:
          # Etiket yoksa boş tensör döndür
          labels = torch.zeros((0, 5))

      # Dönüşümleri uygula
      if self.transform:
          original_img = self.transform(original_img)

      # Düşük çözünürlüklü versiyonu oluştur
      low_res_img = downsample_image(original_img, self.scale_factor)

      # Veri artırma seçeneği
      if self.augment:
          # Rastgele olarak görüntü iyileştirme yöntemlerini uygula
          p = random.random()
          if p < 0.3: # %30 olasılıkla Dark Channel Prior uygula
              low_res_img = apply_dark_channel_prior(low_res_img)
          elif p < 0.6: # %30 olasılıkla CLAHE uygula
              low_res_img = apply_clahe(low_res_img)

      # NumPy dizilerini PyTorch tensörlerine dönüştür
      if isinstance(low_res_img, np.ndarray):
          low_res_img = torch.from_numpy(low_res_img).float()
      elif isinstance(low_res_img, Image.Image):
          # Eğer PIL Image ise, önce numpy'a sonra tensöre dönüştür
          low_res_img = torch.from_numpy(np.array(low_res_img)).permute(2, 0, 1).float() / 255.0

      if isinstance(original_img, np.ndarray):
          original_img = torch.from_numpy(original_img).float()
      elif isinstance(original_img, Image.Image):
          # Eğer PIL Image ise, önce numpy'a sonra tensöre dönüştür
          original_img = torch.from_numpy(np.array(original_img)).permute(2, 0, 1).float() / 255.0

      # Etiketler zaten tensor olduğu için tekrar dönüştürmeye gerek yok

      # Tensörlerin doğru şekilde oluşturulduğunu kontrol et
      assert isinstance(low_res_img, torch.Tensor), "low_res_img must be a PyTorch tensor"
      assert isinstance(original_img, torch.Tensor), "original_img must be a PyTorch tensor"
      assert isinstance(labels, torch.Tensor), "labels must be a PyTorch tensor"

      return low_res_img, original_img, labels

In [43]:
# Eğitim fonksiyonu
def train_sryolo(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device):
    model.to(device)
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        # Eğitim aşaması
        model.train_sr()  # SR modülü eğitim modunda, YOLO dondurulmuş
        train_loss = 0.0

        for batch_idx, (low_res_imgs, original_imgs, labels) in enumerate(train_loader):
            low_res_imgs = low_res_imgs.to(device)
            original_imgs = original_imgs.to(device)

            # Forward pass
            sr_outputs, yolo_results = model(low_res_imgs)

            # Süper çözünürlük kaybını hesapla
            sr_loss = model.compute_sr_loss(sr_outputs, original_imgs)

            # Kaybı güncelle ve geri yayılım yap
            loss = sr_loss  # Burada YOLOv8 kaybı da eklenebilir
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Belirli aralıklarla ilerlemeyi göster
            if (batch_idx + 1) % 10 == 0:
                print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}')

        # Öğrenme oranını güncelle
        scheduler.step()

        # Doğrulama aşaması
        model.eval()  # Değerlendirme modu
        val_loss = 0.0

        with torch.no_grad():
            for low_res_imgs, original_imgs, labels in val_loader:
                low_res_imgs = low_res_imgs.to(device)
                original_imgs = original_imgs.to(device)

                # SR çıktıları ve YOLO sonuçları
                sr_outputs, yolo_results = model(low_res_imgs)

                # SR kaybını hesapla
                sr_loss = model.compute_sr_loss(sr_outputs, original_imgs)
                val_loss += sr_loss.item()

        # Kaybı ortalama al
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

        # En iyi modeli kaydet
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_val_loss,
            }, 'best_sryolo_model.pth')
            print(f'Model kaydedildi (Epoch {epoch+1})')

In [44]:
def test_sryolo(model, test_loader, device, output_dir='outputs'):
    model.to(device)
    model.eval()

    # Çıktı dizinini oluştur
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/low_res", exist_ok=True)
    os.makedirs(f"{output_dir}/sr_output", exist_ok=True)
    os.makedirs(f"{output_dir}/original", exist_ok=True)
    os.makedirs(f"{output_dir}/yolo_result", exist_ok=True)

    # Performans metrikleri
    psnr_values = []
    ssim_values = []
    detection_results = []

    with torch.no_grad():
        for i, (low_res_imgs, original_imgs, labels) in enumerate(test_loader):
            low_res_imgs = low_res_imgs.to(device)
            original_imgs = original_imgs.to(device)

            # Model ile tahmin yap
            sr_outputs, yolo_results = model(low_res_imgs)

            # Sonuçları kaydet ve görselleştir
            for j in range(len(low_res_imgs)):
                # Düşük çözünürlüklü girdiyi kaydet
                lr_img = low_res_imgs[j].cpu().numpy().transpose(1, 2, 0)
                lr_img = (lr_img * 255).clip(0, 255).astype(np.uint8)
                Image.fromarray(lr_img).save(f'{output_dir}/low_res/lr_input_{i}_{j}.jpg')

                # Süper çözünürlük çıktısını kaydet
                sr_img = sr_outputs[j].cpu().numpy().transpose(1, 2, 0)
                sr_img = (sr_img * 255).clip(0, 255).astype(np.uint8)
                Image.fromarray(sr_img).save(f'{output_dir}/sr_output/sr_output_{i}_{j}.jpg')

                # Orijinal görüntüyü kaydet
                orig_img = original_imgs[j].cpu().numpy().transpose(1, 2, 0)
                orig_img = (orig_img * 255).clip(0, 255).astype(np.uint8)
                Image.fromarray(orig_img).save(f'{output_dir}/original/original_{i}_{j}.jpg')

                # YOLOv8 sonuçlarını kaydet
                # Not: yolo_results bir liste değil, bir Results nesnesi olabilir
                if hasattr(yolo_results[j], 'plot'):
                    result_img = yolo_results[j].plot()  # YOLOv8'in sonuç görselleştirme fonksiyonu
                    Image.fromarray(result_img).save(f'{output_dir}/yolo_result/yolo_result_{i}_{j}.jpg')

                # PSNR ve SSIM hesapla
                psnr_value = calculate_psnr(sr_img, orig_img)
                ssim_value = calculate_ssim(sr_img, orig_img)
                psnr_values.append(psnr_value)
                ssim_values.append(ssim_value)

                # Nesne tespiti sonuçlarını kaydet
                if hasattr(yolo_results[j], 'boxes') and hasattr(yolo_results[j].boxes, 'data'):
                    detection_results.append(yolo_results[j].boxes.data.cpu().numpy())
                else:
                    # Eğer yolo_results farklı bir formatta ise buna uyum sağla
                    detection_results.append(np.array([]))

                print(f'Test Görüntüsü {i}_{j} işlendi. PSNR: {psnr_value:.2f}, SSIM: {ssim_value:.4f}')

    # Ortalama metrikleri hesapla
    if psnr_values:
        avg_psnr = sum(psnr_values) / len(psnr_values)
        avg_ssim = sum(ssim_values) / len(ssim_values)
        print(f"\nTest sonuçları:")
        print(f"Ortalama PSNR: {avg_psnr:.2f} dB")
        print(f"Ortalama SSIM: {avg_ssim:.4f}")

        # Nesne tespiti sonuçlarını analiz et
        analyze_detection_results(detection_results, output_dir)

        return avg_psnr, avg_ssim, detection_results
    else:
        print("Hiç test verisi işlenmedi!")
        return 0, 0, []

In [13]:
# PSNR hesaplama fonksiyonu
def calculate_psnr(img1, img2):
    """İki görüntü arasındaki PSNR (Peak Signal-to-Noise Ratio) değerini hesaplar"""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

# SSIM hesaplama fonksiyonu
def calculate_ssim(img1, img2):
    """İki görüntü arasındaki SSIM (Structural Similarity Index) değerini hesaplar"""
    # Gri tonlamalı dönüşüm
    if img1.shape[2] == 3:
        img1_gray = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        img2_gray = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    else:
        img1_gray = img1
        img2_gray = img2

    C1 = (0.01 * 255) ** 2
    C2 = (0.03 * 255) ** 2

    # Ortalama
    mu1 = cv2.GaussianBlur(img1_gray, (11, 11), 1.5)
    mu2 = cv2.GaussianBlur(img2_gray, (11, 11), 1.5)

    # Varyans ve kovaryans
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = cv2.GaussianBlur(img1_gray ** 2, (11, 11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(img2_gray ** 2, (11, 11), 1.5) - mu2_sq
    sigma12 = cv2.GaussianBlur(img1_gray * img2_gray, (11, 11), 1.5) - mu1_mu2

    # SSIM formülü
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean()


In [14]:
# Nesne tespiti sonuçlarını analiz etme fonksiyonu
def analyze_detection_results(detection_results, output_dir):
    """YOLOv8 nesne tespiti sonuçlarını analiz eder ve raporlar"""
    class_counts = {}
    confidence_scores = []
    total_detections = 0

    for detections in detection_results:
        if len(detections) > 0:
            # Her bir tespit için
            for detection in detections:
                class_id = int(detection[5])
                confidence = detection[4]

                # Sınıf sayısını güncelle
                if class_id not in class_counts:
                    class_counts[class_id] = 0
                class_counts[class_id] += 1

                # Güven skorunu ekle
                confidence_scores.append(confidence)
                total_detections += 1

    # Sonuçları yazdır
    with open(f"{output_dir}/detection_analysis.txt", "w") as f:
        f.write("Nesne Tespiti Analizi\n")
        f.write("=====================\n\n")
        f.write(f"Toplam tespit edilen nesne sayısı: {total_detections}\n\n")

        f.write("Sınıf bazında tespit sayıları:\n")
        for class_id, count in class_counts.items():
            f.write(f"Sınıf {class_id}: {count} nesne\n")

        if confidence_scores:
            avg_confidence = sum(confidence_scores) / len(confidence_scores)
            f.write(f"\nOrtalama güven skoru: {avg_confidence:.4f}\n")
            f.write(f"Minimum güven skoru: {min(confidence_scores):.4f}\n")
            f.write(f"Maksimum güven skoru: {max(confidence_scores):.4f}\n")

    print(f"Tespit analizi '{output_dir}/detection_analysis.txt' dosyasına kaydedildi.")

In [45]:
def main():
    # Parametre ve yol tanımları
    yolo_weights = '/content/drive/MyDrive/srcnn_dataset/high_alt_object_detection/training_logs/yolov8_high_alt/weights/best.pt'
    data_dir = '/content/drive/MyDrive/srcnn_dataset/high_alt_object_detection/images/train'
    labels_dir = '/content/drive/MyDrive/srcnn_dataset/high_alt_object_detection/labels/train'
    val_data_dir = '/content/drive/MyDrive/srcnn_dataset/high_alt_object_detection/images/val'
    val_labels_dir = '/content/drive/MyDrive/srcnn_dataset/high_alt_object_detection/labels/val'
    batch_size = 16
    num_epochs = 100
    learning_rate = 0.001
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Cihaz: {device}")

    # Veri dönüşümlerini tanımla
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    # Veri setlerini oluştur
    train_dataset = SRYOLODataset(data_dir, labels_dir, transform=transform, augment=True)
    val_dataset = SRYOLODataset(val_data_dir, val_labels_dir, transform=transform, augment=False)

    # Veri yükleyicilerini oluştur
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # Model oluştur
    try:
        print("Model oluşturuluyor...")
        model = SRYOLO(yolo_weights)
        print("Model başarıyla oluşturuldu!")
    except Exception as e:
        print(f"Model oluşturulurken hata oluştu: {e}")
        return

    # Model yapısını yazdır
    print("SuperResolution Model Yapısı:")
    print(model.sr)

    # Eğitim ayarları
    try:
        # Sadece süper çözünürlük modülünü eğit (YOLOv8 ağırlıkları dondurulmuş olacak)
        params = [p for p in model.sr.parameters() if p.requires_grad]
        optimizer = optim.Adam(params, lr=learning_rate)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

        # Eğitim süreci
        print("Eğitim başlıyor...")
        train_sryolo(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device)

        # Test süreci
        print("Test başlıyor...")
        test_sryolo(model, val_loader, device)

        print("İşlem tamamlandı!")

    except Exception as e:
        print(f"Eğitim veya test sırasında hata oluştu: {e}")
        import traceback
        traceback.print_exc()

In [55]:
if __name__ == "__main__":
    main()

Cihaz: cuda
Model oluşturuluyor...
Model başarıyla oluşturuldu!
SuperResolution Model Yapısı:
SuperResolution(
  (conv1): Conv2d(3, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
  (conv2): Conv2d(64, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv3): Conv2d(32, 3, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
)
Eğitim başlıyor...
Eğitim veya test sırasında hata oluştu: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return c

Traceback (most recent call last):
  File "<ipython-input-45-530841045462>", line 50, in main
    train_sryolo(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device)
  File "<ipython-input-43-07eb7aa5e8bc>", line 11, in train_sryolo
    for batch_idx, (low_res_imgs, original_imgs, labels) in enumerate(train_loader):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 708, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1480, in _next_data
    return self._process_data(data)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1505, in _process_data
    data.reraise()
  File "/usr/local/lib/python3.11/dist-packages/torch/_utils.py", line 733, in reraise
    raise exception
RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (mo

#### Kodlar hatasız çalışıyor fakat her runtime da farklı bir problem çıkıyor. Uğraşmak için yeterince vakit yok. Görüntü İşleme yönteminden devam et, eğer vakit kalırsa tekrar buraya geri dönersin.